# 01 — BDD100K Dataset Setup & Inspection

**Goal:** Extract the official BDD100K archives, verify the full 70K train / 10K val split, and inspect label formats.

> **Note:** This notebook extracts to local Colab SSD for fast inspection. The raw data does NOT persist across sessions. Notebook 02 is self-contained — it re-extracts from the zips (or restores from its cached tar) in its own session.

## Prerequisites

Download these two files from [bdd-data.berkeley.edu](https://bdd-data.berkeley.edu) and place them in your Google Drive:

| File | Google Drive path | Size |
|---|---|---|
| `bdd100k_images_100k.zip` | `EcoCAR/downloads/` | ~6.4 GB |
| `bdd100k_labels.zip` | `EcoCAR/downloads/` | ~100 MB |

## BDD100K Overview

| Item | Detail |
|---|---|
| **Images** | 100,000 driving images (1280x720) |
| **Official split** | 70K train / 10K val / 20K test |
| **Detection classes** | 10: person, rider, car, truck, bus, train, motorcycle, bicycle, traffic light, traffic sign |
| **Additional tasks** | Lane marking (poly2d), drivable area (segmentation masks) |
| **License** | BSD 3-Clause |

## 1 — Install Dependencies

In [1]:
!pip install -q tqdm pyyaml

## 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3 — Path Configuration

In [3]:
import os

# ── Google Drive paths ────────────────────────────────────────────
ECOCAR_ROOT  = "/content/drive/MyDrive/EcoCAR"
DOWNLOADS    = os.path.join(ECOCAR_ROOT, "downloads")

IMAGES_ZIP = os.path.join(DOWNLOADS, "bdd100k_images_100k.zip")
LABELS_ZIP = os.path.join(DOWNLOADS, "bdd100k_labels.zip")

# ── Local extraction target (Colab SSD — fast I/O) ────────────────
BDD_RAW_DIR = "/content/bdd100k_raw"

print("Checking source archives...")
for name, path in [("Images zip", IMAGES_ZIP), ("Labels zip", LABELS_ZIP)]:
    if os.path.isfile(path):
        sz = os.path.getsize(path) / (1024**3)
        print(f"  {name}: {sz:.2f} GB")
    else:
        print(f"  {name}: NOT FOUND at {path}")
        print(f"    -> Download from bdd-data.berkeley.edu and place in {DOWNLOADS}/")

Checking source archives...
  Images zip: 5.28 GB
  Labels zip: 0.18 GB


## 4 -- Inspect Zip Contents

Before extracting, let's see the internal structure of each zip file so we know where files will land.

In [4]:
import zipfile
from collections import defaultdict

for label, zip_path in [("IMAGES", IMAGES_ZIP), ("LABELS", LABELS_ZIP)]:
    if not os.path.isfile(zip_path):
        print(f"{label}: zip not found, skipping inspection")
        continue

    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = zf.namelist()

    print(f"\n{'=' * 60}")
    print(f" {label} ZIP: {os.path.basename(zip_path)}")
    print(f" Total entries: {len(names):,}")
    print(f"{'=' * 60}")

    # Show top-level directories/files
    top_level = defaultdict(int)
    for n in names:
        top = n.split('/')[0]
        top_level[top] += 1
    print("Top-level entries:")
    for k, v in sorted(top_level.items()):
        print(f"  {k}/  ({v:,} files)" if v > 1 else f"  {k}")

    # Show first 20 entries for structure
    print(f"\nFirst 20 entries:")
    for n in names[:20]:
        print(f"  {n}")
    if len(names) > 20:
        print(f"  ... ({len(names) - 20:,} more)")

    # Count by extension
    ext_counts = defaultdict(int)
    for n in names:
        if '.' in n.split('/')[-1]:
            ext = '.' + n.split('/')[-1].rsplit('.', 1)[1].lower()
            ext_counts[ext] += 1
    print(f"\nFile extensions:")
    for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
        print(f"  {ext}: {count:,}")


 IMAGES ZIP: bdd100k_images_100k.zip
 Total entries: 100,004
Top-level entries:
  100k/  (100,004 files)

First 20 entries:
  100k/
  100k/train/
  100k/train/6a9b44bd-91e97262.jpg
  100k/train/4e20e077-db383da3.jpg
  100k/train/5bbdb38f-56be03a1.jpg
  100k/train/05f8b8b5-eb186af1.jpg
  100k/train/1c20ca7a-8af5dea4.jpg
  100k/train/8a5ee740-e1fbb2e7.jpg
  100k/train/234fce8c-8b3d1523.jpg
  100k/train/070b3ecd-efd302c2.jpg
  100k/train/2fef12f6-f31f4da3.jpg
  100k/train/2fd7219c-7ab71764.jpg
  100k/train/4bef10d7-f4763d02.jpg
  100k/train/1f29a419-291b75c8.jpg
  100k/train/5703a746-168f5bb2.jpg
  100k/train/a8c58a5c-51b7c1f3.jpg
  100k/train/67ac4b4f-35064f6b.jpg
  100k/train/13dd9165-bfca3a84.jpg
  100k/train/6461e9dc-878ec17f.jpg
  100k/train/0062298d-fd69d0ec.jpg
  ... (99,984 more)

File extensions:
  .jpg: 100,000

 LABELS ZIP: bdd100k_labels.zip
 Total entries: 100,004
Top-level entries:
  100k/  (100,004 files)

First 20 entries:
  100k/
  100k/train/
  100k/train/6866acb3-cf17e

## 5 -- Extract Archives

Extract both zips to `/content/bdd100k_raw/`. The cells below auto-detect the actual directory structure after extraction.

In [5]:
import zipfile

os.makedirs(BDD_RAW_DIR, exist_ok=True)

for label, zip_path in [("Images", IMAGES_ZIP), ("Labels", LABELS_ZIP)]:
    if not os.path.isfile(zip_path):
        print(f"{label}: zip not found at {zip_path}")
        continue

    # Quick check if already extracted
    already = False
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Check first non-directory entry
        for info in zf.infolist():
            if not info.is_dir():
                test_path = os.path.join(BDD_RAW_DIR, info.filename)
                if os.path.isfile(test_path):
                    already = True
                break

    if already:
        print(f"{label}: already extracted. Skipping.")
        continue

    sz = os.path.getsize(zip_path) / (1024**3)
    print(f"Extracting {label} ({sz:.2f} GB)...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(BDD_RAW_DIR)
    print(f"  Done.")

print("\nExtraction complete.")

Extracting Images (5.28 GB)...
  Done.
Extracting Labels (0.18 GB)...
  Done.

Extraction complete.


## 6 -- Discover Extracted Structure

The zip files may use different internal layouts. This cell auto-detects where images and labels ended up.

In [6]:
import glob

print("=" * 60)
print(" DISCOVERING EXTRACTED STRUCTURE")
print("=" * 60)

# Show top-level contents
print(f"\nContents of {BDD_RAW_DIR}/:")
for item in sorted(os.listdir(BDD_RAW_DIR)):
    full = os.path.join(BDD_RAW_DIR, item)
    if os.path.isdir(full):
        sub = os.listdir(full)
        print(f"  {item}/  ({len(sub)} items)")
        for s in sorted(sub)[:10]:
            sf = os.path.join(full, s)
            if os.path.isdir(sf):
                print(f"    {s}/  ({len(os.listdir(sf))} items)")
            else:
                print(f"    {s}")
        if len(sub) > 10:
            print(f"    ... ({len(sub)-10} more)")
    else:
        sz = os.path.getsize(full) / (1024**2)
        print(f"  {item}  ({sz:.1f} MB)")

# ── Auto-detect image directories ────────────────────────────────
# Search for directories named "train" or "val" that contain .jpg files
TRAIN_IMG_DIR = None
VAL_IMG_DIR = None

candidate_patterns = [
    # Common BDD100K structures
    os.path.join(BDD_RAW_DIR, "bdd100k", "images", "100k", "{split}"),
    os.path.join(BDD_RAW_DIR, "images", "100k", "{split}"),
    os.path.join(BDD_RAW_DIR, "100k", "{split}"),
    os.path.join(BDD_RAW_DIR, "bdd100k", "images", "{split}"),
    os.path.join(BDD_RAW_DIR, "images", "{split}"),
    os.path.join(BDD_RAW_DIR, "{split}"),
]

for pattern in candidate_patterns:
    train_cand = pattern.format(split="train")
    val_cand = pattern.format(split="val")
    if os.path.isdir(train_cand):
        # Check if it has images
        sample = [f for f in os.listdir(train_cand)[:20] if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if sample:
            TRAIN_IMG_DIR = train_cand
            VAL_IMG_DIR = val_cand if os.path.isdir(val_cand) else None
            break

print(f"\n{'=' * 60}")
print(f" IMAGE DIRECTORIES FOUND")
print(f"{'=' * 60}")
if TRAIN_IMG_DIR:
    n_train = sum(1 for f in os.listdir(TRAIN_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    print(f"  Train: {TRAIN_IMG_DIR}  ({n_train:,} images)")
else:
    print("  Train: NOT FOUND")

if VAL_IMG_DIR and os.path.isdir(VAL_IMG_DIR):
    n_val = sum(1 for f in os.listdir(VAL_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    print(f"  Val:   {VAL_IMG_DIR}  ({n_val:,} images)")
else:
    print("  Val:   NOT FOUND")

# Also check for test
for pattern in candidate_patterns:
    test_cand = pattern.format(split="test")
    if os.path.isdir(test_cand):
        sample = [f for f in os.listdir(test_cand)[:5] if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if sample:
            n_test = sum(1 for f in os.listdir(test_cand) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
            print(f"  Test:  {test_cand}  ({n_test:,} images)")
            break

print(f"\nExpected: ~70K train / ~10K val / ~20K test")

 DISCOVERING EXTRACTED STRUCTURE

Contents of /content/bdd100k_raw/:
  100k/  (3 items)
    test/  (40000 items)
    train/  (140000 items)
    val/  (20000 items)

 IMAGE DIRECTORIES FOUND
  Train: /content/bdd100k_raw/100k/train  (70,000 images)
  Val:   /content/bdd100k_raw/100k/val  (10,000 images)
  Test:  /content/bdd100k_raw/100k/test  (20,000 images)

Expected: ~70K train / ~10K val / ~20K test


## 7 -- Discover Label Files

Auto-detect label files regardless of zip internal structure.

In [7]:
print("=" * 60)
print(" ALL JSON LABEL FILES")
print("=" * 60)

# Find ALL JSON files under BDD_RAW_DIR
all_jsons = sorted(glob.glob(os.path.join(BDD_RAW_DIR, "**", "*.json"), recursive=True))
for f in all_jsons:
    sz = os.path.getsize(f) / (1024**2)
    rel = os.path.relpath(f, BDD_RAW_DIR)
    print(f"  {rel:.<60} {sz:>8.1f} MB")

if not all_jsons:
    print("  No JSON files found!")
    print(f"  Looking in: {BDD_RAW_DIR}")

# Find all directories that might contain labels
print(f"\nAll directories under {BDD_RAW_DIR}:")
for root, dirs, files in os.walk(BDD_RAW_DIR):
    depth = root.replace(BDD_RAW_DIR, '').count(os.sep)
    if depth > 3:  # Don't recurse too deep
        continue
    indent = '  ' * depth
    rel = os.path.relpath(root, BDD_RAW_DIR)
    n_files = len(files)
    n_dirs = len(dirs)
    if rel == '.':
        continue
    print(f"  {indent}{os.path.basename(root)}/  ({n_files} files, {n_dirs} subdirs)")

流式输出内容被截断，只能显示最后 5000 行内容。
  100k/val/be769e9f-abea5fd9.json.............................      0.0 MB
  100k/val/be769e9f-ed37df2b.json.............................      0.0 MB
  100k/val/be76ea62-78b2c8f8.json.............................      0.0 MB
  100k/val/be776742-fe6599b2.json.............................      0.0 MB
  100k/val/be777044-7db298a4.json.............................      0.0 MB
  100k/val/be777044-db29e596.json.............................      0.0 MB
  100k/val/be7783b0-b44ff154.json.............................      0.0 MB
  100k/val/be791a81-a6b53fef.json.............................      0.0 MB
  100k/val/be7b856e-d1ece683.json.............................      0.0 MB
  100k/val/be7c5bf9-f31d60c1.json.............................      0.0 MB
  100k/val/be80a253-75e6f6aa.json.............................      0.0 MB
  100k/val/be819732-9cff62ba.json.............................      0.0 MB
  100k/val/be832d0b-0ed20122.json.............................      0.0 M

## 8 -- Locate Detection Labels

BDD100K detection labels come in several possible formats and locations. This cell searches for them automatically.

In [8]:
import json

# Build comprehensive candidate list by searching all found JSONs
train_label_candidates = []
val_label_candidates = []

for jf in all_jsons:
    bn = os.path.basename(jf).lower()
    # Detection label files
    if 'train' in bn and ('det' in bn or 'label' in bn):
        train_label_candidates.append(jf)
    elif 'val' in bn and ('det' in bn or 'label' in bn):
        val_label_candidates.append(jf)

# Also add hard-coded known paths
for prefix in [
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels"),
    os.path.join(BDD_RAW_DIR, "labels"),
    BDD_RAW_DIR,
]:
    for sub in ["det_20", "det_train", ""]:
        train_label_candidates.append(os.path.join(prefix, sub, "det_train.json") if sub else os.path.join(prefix, "det_train.json"))
        val_label_candidates.append(os.path.join(prefix, sub, "det_val.json") if sub else os.path.join(prefix, "det_val.json"))
    train_label_candidates.append(os.path.join(prefix, "bdd100k_labels_images_train.json"))
    val_label_candidates.append(os.path.join(prefix, "bdd100k_labels_images_val.json"))

# Find first existing file for each split
det_label_paths = {}
for split, candidates in [("train", train_label_candidates), ("val", val_label_candidates)]:
    for c in candidates:
        if os.path.isfile(c):
            det_label_paths[split] = c
            sz = os.path.getsize(c) / (1024**2)
            print(f"  {split} det labels: {os.path.relpath(c, BDD_RAW_DIR)} ({sz:.1f} MB)")
            break
    else:
        print(f"  {split} det labels: NOT FOUND")

if len(det_label_paths) < 2:
    print("\nWARNING: Could not find both train and val detection label files!")
    print("Available JSON files:")
    for jf in all_jsons:
        print(f"  {os.path.relpath(jf, BDD_RAW_DIR)}")
    print("\nYou may need to download detection labels separately from bdd-data.berkeley.edu")
else:
    print("\nBoth detection label files found.")

流式输出内容被截断，只能显示最后 5000 行内容。
  100k/val/be731030-7593da2a.json
  100k/val/be731030-82d58159.json
  100k/val/be731030-c29affd6.json
  100k/val/be73806a-cd8633af.json
  100k/val/be769e9f-abea5fd9.json
  100k/val/be769e9f-ed37df2b.json
  100k/val/be76ea62-78b2c8f8.json
  100k/val/be776742-fe6599b2.json
  100k/val/be777044-7db298a4.json
  100k/val/be777044-db29e596.json
  100k/val/be7783b0-b44ff154.json
  100k/val/be791a81-a6b53fef.json
  100k/val/be7b856e-d1ece683.json
  100k/val/be7c5bf9-f31d60c1.json
  100k/val/be80a253-75e6f6aa.json
  100k/val/be819732-9cff62ba.json
  100k/val/be832d0b-0ed20122.json
  100k/val/be832d0b-2384fe4c.json
  100k/val/be83776d-e07b8fdc.json
  100k/val/be839b4c-fba3ea45.json
  100k/val/be83fbfa-53a6b842.json
  100k/val/be85292c-a3fbba40.json
  100k/val/be852dd5-29511af0.json
  100k/val/be854092-492e5e65.json
  100k/val/be85665c-4fe33b5c.json
  100k/val/be8590fa-52948d8f.json
  100k/val/be8590fa-9ab10ced.json
  100k/val/be85bd88-4aad89d7.json
  100k/val/be85bd88-d

## 9 — Inspect Label Format

In [9]:
if len(det_label_paths) == 2:
    # Load and inspect train labels
    with open(det_label_paths["train"], 'r') as f:
        train_labels = json.load(f)

    print(f"Train labels: {len(train_labels):,} frames")
    if train_labels:
        print(f"First entry keys: {list(train_labels[0].keys())}")
        print(f"\nSample entry (truncated):")
        print(json.dumps(train_labels[0], indent=2)[:1500])

    # Count detection categories
    categories = set()
    box_count = 0
    lane_count = 0
    for frame in train_labels:
        for lbl in (frame.get('labels') or []):
            categories.add(lbl.get('category', ''))
            if 'box2d' in lbl:
                box_count += 1
            if 'poly2d' in lbl:
                lane_count += 1

    print(f"\nCategories found: {sorted(categories)}")
    print(f"Total box2d annotations: {box_count:,}")
    print(f"Total poly2d annotations: {lane_count:,}")

    # Val stats
    with open(det_label_paths["val"], 'r') as f:
        val_labels = json.load(f)
    print(f"\nVal labels: {len(val_labels):,} frames")
else:
    # Inspect whatever JSON we can find
    if all_jsons:
        print("Inspecting first available JSON file:")
        with open(all_jsons[0], 'r') as f:
            sample = json.load(f)
        print(f"  File: {os.path.relpath(all_jsons[0], BDD_RAW_DIR)}")
        print(f"  Type: {type(sample).__name__}")
        if isinstance(sample, list):
            print(f"  Length: {len(sample)}")
            if sample:
                print(f"  First entry keys: {list(sample[0].keys()) if isinstance(sample[0], dict) else type(sample[0])}")
                print(json.dumps(sample[0], indent=2)[:1000])
        elif isinstance(sample, dict):
            print(f"  Keys: {list(sample.keys())[:20]}")
            print(json.dumps(sample, indent=2)[:1000])
    else:
        print("No JSON files found to inspect.")

Inspecting first available JSON file:
  File: 100k/test/cabc30fc-e7726578.json
  Type: dict
  Keys: ['name', 'frames', 'attributes']
{
  "name": "cabc30fc-e7726578",
  "frames": [
    {
      "timestamp": 10000,
      "objects": [
        {
          "category": "traffic sign",
          "id": 0,
          "attributes": {
            "occluded": false,
            "truncated": false,
            "trafficLightColor": "none"
          },
          "box2d": {
            "x1": 574.41345,
            "y1": 376.718573,
            "x2": 600.772767,
            "y2": 415.159242
          }
        },
        {
          "category": "traffic sign",
          "id": 1,
          "attributes": {
            "occluded": false,
            "truncated": false,
            "trafficLightColor": "none"
          },
          "box2d": {
            "x1": 653.4914,
            "y1": 404.176194,
            "x2": 666.671061,
            "y2": 422.847376
          }
        },
        {
          "categor

## 10 — Save Paths Config

Save a `paths_config.yaml` so notebook 02 can locate the extracted data without guessing.

In [10]:
import yaml

assert TRAIN_IMG_DIR is not None, "Could not find train images directory! Check extraction output above."

paths_config = {
    "bdd_raw_dir": BDD_RAW_DIR,
    "train_images_dir": TRAIN_IMG_DIR,
    "val_images_dir": VAL_IMG_DIR or "",
}
if det_label_paths.get("train"):
    paths_config["det_train_json"] = det_label_paths["train"]
if det_label_paths.get("val"):
    paths_config["det_val_json"] = det_label_paths["val"]

config_path = os.path.join(ECOCAR_ROOT, "paths_config.yaml")
with open(config_path, 'w') as f:
    yaml.dump(paths_config, f, default_flow_style=False)

print(f"Saved: {config_path}")
for k, v in paths_config.items():
    print(f"  {k}: {v}")

Saved: /content/drive/MyDrive/EcoCAR/paths_config.yaml
  bdd_raw_dir: /content/bdd100k_raw
  train_images_dir: /content/bdd100k_raw/100k/train
  val_images_dir: /content/bdd100k_raw/100k/val


## 11 — Summary

The official BDD100K dataset is now extracted on local Colab SSD for fast I/O.

In [11]:
print("\n" + "=" * 60)
print(" NB01 COMPLETE")
print("=" * 60)
print(f"  Raw data dir:   {BDD_RAW_DIR}")
print(f"  Train images:   {TRAIN_IMG_DIR}")
print(f"  Val images:     {VAL_IMG_DIR or 'NOT FOUND'}")
if det_label_paths.get("train"):
    print(f"  Train labels:   {det_label_paths['train']}")
if det_label_paths.get("val"):
    print(f"  Val labels:     {det_label_paths['val']}")
print(f"  Paths config:   {config_path}")
print("=" * 60)
print("\nNext: Run 02_bdd100k_preparation.ipynb to convert to YOLO format.")


 NB01 COMPLETE
  Raw data dir:   /content/bdd100k_raw
  Train images:   /content/bdd100k_raw/100k/train
  Val images:     /content/bdd100k_raw/100k/val
  Paths config:   /content/drive/MyDrive/EcoCAR/paths_config.yaml

Next: Run 02_bdd100k_preparation.ipynb to convert to YOLO format.


---

### Notes

- **Official splits preserved:** We use the train/val split as provided by Berkeley. No fabricated splits.
- **Test images** are extracted but have no public labels (BDD100K benchmark submission only). The website provides info on test set evaluation.
- **Lane & drivable labels** (if present in the labels zip) will be used in notebook 07.
- **Local SSD only:** The raw data extracted here lives on this session's local SSD and will NOT persist. Notebook 02 is self-contained — it re-extracts from the original zips on Drive (or restores from its own cached YOLO tar).
- The `paths_config.yaml` saved to Drive records the discovered structure so NB02 knows which candidate paths to try.